# Stage-wise head specialization

Fetch runs from W&B and read off **which attention head learned which context
position, and in what order** — then check that order against the one each run's
teacher config predicts.

Logic lives in `src/analysis/head_phases.py`; this notebook is the shell around it.
Run from the repository root.

In [ ]:
%load_ext autoreload
%autoreload 2

# Jupyter starts the kernel in this notebook's directory, so put the repo root on
# sys.path — otherwise `import src` fails depending on where you launched from.
import pathlib, sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.analysis.head_phases import (
    COS_SIM, DIFFUSE, SPAN_MASS, VALUE_COS, VALUE_INNER,
    analyze_run, head_span_series, metric_keys, phase_table, span_offsets,
    run_label, short_axis_names, varying_config_columns,
)
from notebooks.report_head_phases import DEFAULT_ENTITY, DEFAULT_PROJECT, _demo_frame

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

## 1. Configure

`DEMO = True` runs on a synthetic frame so the whole notebook works with no W&B
access — flip it to `False` to pull real runs.

`SPAN_LENGTHS` must match the teacher the runs were trained with. With
`[1, 1, 1]` each span is a single token, so span `k` is offset `-(W-k)` and the
labels below read `-3 / -2 / -1`.

In [ ]:
DEMO = False

TAGS = ["power-ablation"] #["power-spectrum"]
ENTITY, PROJECT = DEFAULT_ENTITY, DEFAULT_PROJECT

NUM_HEADS = 3            # student.num_heads
NUM_SPANS = 3            # teacher.window
SPAN_LENGTHS = [1, 1, 1] # teacher.span_lengths
LAYER, SPLIT = "L1", "train"

# Two independent read-outs of the same story:
#   SPAN_MASS / COS_SIM     - WHERE a head looks (attention over context positions)
#   VALUE_COS / VALUE_INNER - WHICH teacher lag matrix its value projection
#                             implements (needs attention_disentanglement=true)
# Section 6 cross-checks them; disagreement means the read-out is unreliable.
METRIC = SPAN_MASS

# A head counts as *focused* on a span when that span clears CUTOFF under
# STRATEGY. Heads are judged independently, so several may share a span and some
# spans may go uncovered — there is no matching.
#   share    - span's fraction of the head's mass  (uniform = 1/K, so >1/K)
#   absolute - raw value; natural for align_cos_sim (try 0.7)
#   margin   - gap to the runner-up, raw units
#   ratio    - best / runner-up; scale-free (try 2.0)
CUTOFF, STRATEGY = 0.5, "share"

# Samples a label must persist before it counts as a phase (filters flicker).
MIN_DWELL = 2

LABELS = span_offsets(SPAN_LENGTHS)
LABELS

## 2. Fetch

`get_runs_data` returns history **and** flattened config (`cfg.*`) columns, which
is what lets the next step compare observed order against the configured one.

Note `utils.py`'s module-level defaults point at the *previous* repo's project;
a wrong target returns an empty frame silently, so entity/project are explicit.

In [ ]:
if DEMO:
    df = _demo_frame(NUM_HEADS, NUM_SPANS)
    print(f"demo frame: {df._run_id.nunique()} runs, {len(df)} rows")
else:
    from notebooks.utils import fetch_runs, get_runs_data

    runs = fetch_runs(entity=ENTITY, project=PROJECT, tags_any=TAGS)
    print(f"{len(runs)} run(s) tagged {TAGS} in {ENTITY}/{PROJECT}")

    keys = metric_keys(NUM_HEADS, NUM_SPANS, layer=LAYER, split=SPLIT, metric=METRIC)
    df = get_runs_data(runs, keys)
    assert not df.empty, (
        f"No history row carried all {len(keys)} attention keys. Check that the runs "
        "used a TransformerDecoder student with misc.log_attention_frequency set, and "
        "that NUM_HEADS/NUM_SPANS match."
    )
    print(f"{df._run_id.nunique()} runs, {len(df)} rows")

## 3. The story, one row per run

`summary` is the headline: `H3:-1@50 -> H2:-2@400 -> H1:-3@800` reads as
*head 3 locked onto offset -1 at step 50, then head 2 onto -2 at 400, then head 1
onto -3 at 800*.

`head{h}_path` is the migration story: `diffuse@0 > -1@150 > -3@600` means head
*h* was uncommitted, focused on offset -1 at step 150, then moved to -3 at 600.
`migrations` counts those moves; `shared_spans` counts spans two heads ended up
on; `uncovered_spans` counts spans nobody took.

`predicted_order` comes from the run's own `lag_spectrum` — the teacher's outer
importance law — so `order_matches` tests the hypothesis that spans are covered
in decreasing teacher importance. `order_rank_corr` is the Spearman version
(+1 exact agreement, -1 exactly reversed).

In [ ]:
# group_cols is left unset: phase_table discovers every config column that
# VARIES across the fetched runs, so whatever the sweep changed (alpha_in,
# alpha_out, rank, orthogonality, num_heads, seed, ...) appears automatically.
print("sweep axes:", varying_config_columns(df))

table = phase_table(
    df,
    num_heads=NUM_HEADS,
    num_spans=NUM_SPANS,
    span_lengths=SPAN_LENGTHS,
    layer=LAYER,
    split=SPLIT,
    metric=METRIC,
    cutoff=CUTOFF,
    strategy=STRATEGY,
    min_dwell=MIN_DWELL,
)
# run id -> the axes that distinguish it, for use as plot titles below.
LABELS_BY_RUN = dict(zip(table["_run_id"], table["label"]))

table.drop(columns=["_run_id"])

## 4. Trajectories

One panel per span, one line per head. The dashed vertical line marks the
acquisition step of the head that *owns* that span. A clean staged run shows the
owning head rising well clear of the others, with the dashed lines ordered
left-to-right by teacher importance.

In [ ]:
def plot_run(run_df, run_label=None, metric=METRIC, ax_row=None):
    """Per-span attention trajectories with acquisition markers."""
    steps, values = head_span_series(
        run_df, NUM_HEADS, NUM_SPANS, layer=LAYER, split=SPLIT, metric=metric
    )
    phases = analyze_run(
        run_df, NUM_HEADS, NUM_SPANS, span_lengths=SPAN_LENGTHS,
        layer=LAYER, split=SPLIT, metric=metric,
        cutoff=CUTOFF, strategy=STRATEGY, min_dwell=MIN_DWELL,
    )
    owners = phases.final_owners   # span -> [heads]; may be several or none

    axes = ax_row if ax_row is not None else plt.subplots(
        1, NUM_SPANS, figsize=(4 * NUM_SPANS, 3.2), sharey=True
    )[1]

    for k in range(NUM_SPANS):
        ax = axes[k]
        for h in range(NUM_HEADS):
            owns = h in owners[k]
            ax.plot(
                steps, values[:, h, k],
                color=series_color(h),
                linewidth=2.4 if owns else 1.2,
                alpha=1.0 if owns else 0.45,
                label=f"head {h + 1}",
            )
        step = phases.acquired.get(k)
        if step is not None:
            ax.axvline(step, color="0.35", linestyle="--", linewidth=1.2)
            ax.text(step, ax.get_ylim()[1], f" {step:g}", va="top", fontsize=8, color="0.35")
        ax.set_title(f"span {k + 1}  (offset {LABELS[k]})", fontsize=10)
        ax.set_xlabel("step")
        ax.spines[["top", "right"]].set_visible(False)
    axes[0].set_ylabel(metric)
    axes[0].legend(frameon=False, fontsize=8)
    if run_label:
        axes[0].figure.suptitle(run_label, fontsize=11, y=1.02)
    return phases


# Fixed, non-cycled order so a head keeps its colour across every panel and run.
# Validated with the palette checker (all-pairs, light AND dark): worst CVD
# separation dE 11.4, normal-vision 24.2 — both clear of the floors. Do not
# extend by inventing a 4th hue: no 4-way categorical set passed all-pairs here.
# Beyond 3 series, facet instead (or lean on the ribbon's direct labels).
SERIES_COLORS = ["#1b6ca8", "#d95f02", "#009E73"]
NEUTRAL = "0.55"


def series_color(i):
    """Validated hue for the first 3 series; neutral beyond, where the direct
    labels carry identity instead of colour."""
    return SERIES_COLORS[i] if i < len(SERIES_COLORS) else NEUTRAL


HEAD_COLORS = SERIES_COLORS

run_ids = list(df["_run_id"].unique())
fig, grid = plt.subplots(
    len(run_ids), NUM_SPANS,
    figsize=(4 * NUM_SPANS, 3.2 * len(run_ids)),
    sharey=True, squeeze=False,
)
for row, rid in enumerate(run_ids):
    sub = df[df["_run_id"] == rid]
    p = plot_run(sub, ax_row=grid[row])
    grid[row][0].set_ylabel(f"{METRIC}", fontsize=9)
    grid[row][0].annotate(
        LABELS_BY_RUN.get(rid, sub["_run_name"].iloc[0]),
        xy=(0, 1.22), xycoords="axes fraction", fontsize=10, fontweight="bold",
    )
    print(f"{LABELS_BY_RUN.get(rid, ''):<40s}  {p.summary}")
plt.tight_layout()
plt.show()

## 4b. Phase ribbons — the migration story

One row per head, time along x, colour = the span that head is focused on.
Grey is `diffuse` (uncommitted). This is where you see the collaborative
regime: all heads grey or all on the *same* colour early, then breaking away
into distinct colours. Two rows sharing a colour at the right edge means two
heads ended on the same span.

In [ ]:
def plot_ribbons(run_df, ax, metric=METRIC):
    phases = analyze_run(
        run_df, NUM_HEADS, NUM_SPANS, span_lengths=SPAN_LENGTHS,
        layer=LAYER, split=SPLIT, metric=metric,
        cutoff=CUTOFF, strategy=STRATEGY, min_dwell=MIN_DWELL,
    )
    # Sequential-by-identity: one fixed colour per span, grey reserved for diffuse.
    span_colors = {k: series_color(k) for k in range(NUM_SPANS)}
    span_colors[DIFFUSE] = "0.85"

    for h, segments in enumerate(phases.trajectories):
        for seg in segments:
            ax.barh(
                h, seg.end_step - seg.start_step, left=seg.start_step, height=0.62,
                color=span_colors[seg.span],
                edgecolor="white", linewidth=1.5,   # 2px-ish surface gap between phases
            )
            if seg.span != DIFFUSE and seg.end_step - seg.start_step > 0:
                ax.text(
                    (seg.start_step + seg.end_step) / 2, h, LABELS[seg.span],
                    ha="center", va="center", fontsize=8, color="white", fontweight="bold",
                )
    ax.set_yticks(range(NUM_HEADS), [f"head {h + 1}" for h in range(NUM_HEADS)])
    ax.invert_yaxis()
    ax.set_xlabel("step")
    ax.spines[["top", "right", "left"]].set_visible(False)
    return phases


fig, axes = plt.subplots(
    len(run_ids), 1, figsize=(11, 1.1 * NUM_HEADS * len(run_ids)), squeeze=False
)
for row, rid in enumerate(run_ids):
    sub = df[df["_run_id"] == rid]
    p = plot_ribbons(sub, axes[row][0])
    axes[row][0].set_title(
        f"{LABELS_BY_RUN.get(rid, sub['_run_name'].iloc[0])}   |   "
        f"{p.migrations} migration(s), {p.shared_spans} shared, "
        f"{p.uncovered_spans} uncovered",
        fontsize=10, loc="left",
    )
plt.tight_layout()
plt.show()

## 5. Does the sweep's importance law predict the learning order?

Across an $\alpha_{in} \times \alpha_{out}$ grid, `order_rank_corr` per cell.
`+1` means heads were acquired exactly in decreasing teacher importance.

This is the payoff plot for the power-law ablations: it says whether the *shape*
of the teacher's spectrum controls the *order* of head specialization.

In [ ]:
A_IN, A_OUT = "cfg.teacher.spectrum.alpha", "cfg.teacher.lag_spectrum.alpha"
AXES = [c for c in table.columns if c.startswith("cfg.")]

if {A_IN, A_OUT}.issubset(table.columns) and table[A_IN].nunique() > 1:
    grid_df = table.pivot_table(index=A_IN, columns=A_OUT, values="order_rank_corr")
    fig, ax = plt.subplots(figsize=(1.1 * len(grid_df.columns) + 2, 1.0 * len(grid_df) + 1.5))
    # Diverging: polarity around 0, neutral midpoint, symmetric limits.
    im = ax.imshow(grid_df.values, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(grid_df.columns)), grid_df.columns)
    ax.set_yticks(range(len(grid_df.index)), grid_df.index)
    ax.set_xlabel(r"$\alpha_{out}$  (position importance decay)")
    ax.set_ylabel(r"$\alpha_{in}$  (feature importance decay)")
    ax.set_title("order agreement with the teacher's importance law")
    for i in range(len(grid_df.index)):
        for j in range(len(grid_df.columns)):
            v = grid_df.values[i, j]
            if np.isfinite(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9,
                        color="white" if abs(v) > 0.6 else "0.2")
    fig.colorbar(im, ax=ax, label="Spearman rank corr", shrink=0.8)
    plt.tight_layout()
    plt.show()
else:
    print("This heatmap needs a 2-D sweep over both alphas.\n"
          f"Axes that actually vary here: {AXES or 'none — all runs share one config'}\n"
          "For a 1-D sweep (e.g. the spine), read section 5b instead: it plots\n"
          "coverage step against the realized lag weight, which works for any law.")

## 5b. Coverage time vs the teacher's realized importance

`phase_table` derives `lag_weight_*` per run by re-evaluating the run's own
`lag_spectrum` block through `src.spectra` — nothing extra is logged, because
these weights depend only on the config (the RNG touches singular *vectors*,
never the singular values or norms). Deriving rather than logging keeps one
source of truth and works on runs that predate any of this.

It matters because `alpha` and `decay` are not comparable numbers across laws —
a geometric `decay=1.7` and a power `alpha=1.0` only line up through the weights
they generate.

So this plot puts every run on one physical axis: a lag's importance against the
step at which it was covered. If acquisition is importance-driven, points fall on
a decreasing trend regardless of which law produced them.

In [ ]:
W_COL = "lag_weight_{label}"

points = []
for _, row in table.iterrows():
    for k in range(NUM_SPANS):
        w = row.get(W_COL.format(label=LABELS[k]))
        if w is None or pd.isna(w):
            continue
        points.append({"weight": float(w), "step": row.get(f"acq_{LABELS[k]}"),
                       "span": LABELS[k], "run": row["label"]})
pts = pd.DataFrame(points)

if pts.empty:
    print("No lag weights derived — these runs have no cfg.teacher.lag_spectrum.*\n"
          "block (a non-linear teacher, or a config predating the spectrum work).")
else:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    done = pts.dropna(subset=["step"])
    top = (done["step"].max() * 1.6) if len(done) and done["step"].max() > 0 else 1.0
    for i, span in enumerate(LABELS):
        grp = pts[pts["span"] == span]
        hit, miss = grp.dropna(subset=["step"]), grp[grp["step"].isna()]
        ax.scatter(hit["weight"], hit["step"], s=70, color=series_color(i),
                   edgecolor="white", linewidth=1.5, label=f"offset {span}", zorder=3)
        if len(miss):   # never covered: park on the top edge rather than drop
            ax.scatter(miss["weight"], [top] * len(miss), s=70, marker="^",
                       color=series_color(i), edgecolor="white", linewidth=1.5, zorder=3)
    ax.set_xscale("log")
    ax.set_yscale("symlog")
    ax.set_xlabel("teacher lag weight (realized matrix norm)")
    ax.set_ylabel("coverage step")
    ax.set_title("more important lags should be covered earlier")
    # Name each run once, at its highest-weight point, so a cloud of dots is
    # traceable back to the config that produced it.
    for run, grp in pts.dropna(subset=["step"]).groupby("run"):
        tip = grp.loc[grp["weight"].idxmax()]
        ax.annotate(run, (tip["weight"], tip["step"]), fontsize=7, color="0.35",
                    xytext=(4, 4), textcoords="offset points")
    ax.legend(frameon=False, fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.show()
    print("^ markers = never covered within the run")

## 6. Cross-check the metric

`span_mass` and `align_cos_sim` are the same quantity raw vs scale-free.
`value_cos` is the independent one: attention says *where* a head looks, value
alignment says *which* teacher lag matrix its value projection implements.

If they disagree on which lag a head owns, treat the read-out as unreliable for
that run and look at the trajectories directly. `value_cos` needs the student's
`attention_disentanglement=true` (per-head value projections); metrics whose keys
are absent for a run come back as `None`.

In [ ]:
for rid in run_ids:
    sub = df[df["_run_id"] == rid]
    name = sub["_run_name"].iloc[0]
    out = {}
    for metric in (SPAN_MASS, COS_SIM, VALUE_COS):
        try:
            out[metric] = analyze_run(
                sub, NUM_HEADS, NUM_SPANS, span_lengths=SPAN_LENGTHS,
                layer=LAYER, split=SPLIT, metric=metric,
                cutoff=CUTOFF, strategy=STRATEGY, min_dwell=MIN_DWELL,
            ).assignment
        except KeyError:
            out[metric] = None
    present = {k: v for k, v in out.items() if v is not None}
    agree = len({tuple(v) for v in present.values()}) == 1 if len(present) > 1 else "n/a"
    shown = "  ".join(f"{k}={v}" for k, v in out.items())
    print(f"{LABELS_BY_RUN.get(rid, name):<40s}  {shown}  agree={agree}")